In [ ]:
!pip install -q torch transformers datasets matplotlib

In [ ]:
import math
import re
import time
from collections import Counter

import matplotlib.pyplot as plt
import torch
import transformers
from datasets import load_dataset
from transformers import (
    AutoModelForCausalLM,
    AutoModelForSequenceClassification,
    AutoTokenizer,
)

DEVICE = torch.device(
    "cuda"
    if torch.cuda.is_available()
    else "mps" if torch.backends.mps.is_available() else "cpu"
)
MODEL_NAME = "Qwen/Qwen3-0.6B"
REWARD_MODEL_NAME = "Skywork/Skywork-Reward-V2-Qwen3-0.6B"
SEED = 0

torch.manual_seed(SEED)
transformers.logging.set_verbosity_error()
print(f"Using device: {DEVICE}")

# Test-Time Compute and Reasoning Control

---
# 1) Introduction

A trained model's accuracy is not fixed: it can be raised at inference by **spending more
compute per query**. There are two axes:

- **Sequential.** Let one sample think longer: a reasoning trace before the answer, extended or
  cut by a budget, or revised in further rounds.
- **Parallel.** Draw $N$ independent samples and aggregate them by majority vote or with a
  verifier.

Reasoning models made the sequential axis a product feature: `reasoning_effort` in the OpenAI
API, `thinking.budget_tokens` in Claude, `thinkingBudget` in Gemini, `enable_thinking` in Qwen3.
Serving them well means controlling how much they think, because token cost and latency grow with
the trace while accuracy saturates.

| Section | What is built and measured |
|---|---|
| 4.1 Budget forcing | s1-style control of the thinking length: forced stop at a budget, "Wait" extension |
| 4.2 Thinking on/off | Qwen3's hybrid thinking switch and the accuracy-vs-budget curve |
| 4.3 Parallel sampling | Majority vote, best-of-N with a reward model, weighted vote, oracle pass@N |
| 4.4 Sequential revision | Self-refinement without external feedback, with flip rates |
| 4.5 Matched compute | Accuracy vs generated tokens across all strategies |
| 4.6 Overthinking | Where the answer is first reached inside a trace; answer-convergence early exit |

Everything runs on `Qwen3-0.6B`, a hybrid model with thinking and non-thinking modes in one
checkpoint, on the 16 GSM8K test problems with the shortest reference solutions. A 0.6B model's thinking
traces on these mostly fit in 768 tokens, which keeps every cell to a few minutes on a laptop; on
full GSM8K the same experiments need 4K-token budgets.

---
# 2) Setup

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, padding_side="left")
model = (
    AutoModelForCausalLM.from_pretrained(MODEL_NAME, dtype=torch.bfloat16)
    .to(DEVICE)
    .eval()
)
THINK_END = "</think>"
THINK_END_ID = tokenizer.convert_tokens_to_ids(THINK_END)
print(
    f"{MODEL_NAME}: {sum(p.numel() for p in model.parameters()) / 1e9:.2f}B params, "
    f"</think> id {THINK_END_ID}"
)

NUM_PROBLEMS = 16
test = load_dataset("openai/gsm8k", "main", split="test")
# the problems with the shortest reference solutions, whose thinking traces mostly fit in 768 tokens
easiest = sorted(range(len(test)), key=lambda i: len(test[i]["answer"]))[:NUM_PROBLEMS]
gsm8k = test.select(easiest)
questions = gsm8k["question"]
golds = [float(a.split("####")[-1].replace(",", "")) for a in gsm8k["answer"]]
INSTRUCTION = "Please reason step by step, and put your final answer within \\boxed{}."
print(f"{NUM_PROBLEMS} problems; first gold answer {golds[0]}")

`prompt` renders the real Qwen3 chat template. With `enable_thinking=False` the template
pre-fills an empty `<think>\n\n</think>` block, so the model answers directly. `boxed_answer`
parses the format the instruction asks for.

In [ ]:
def prompt(question: str, thinking: bool) -> str:
    """Chat-templated prompt in thinking or non-thinking mode."""
    return tokenizer.apply_chat_template(
        [{"role": "user", "content": f"{question}\n{INSTRUCTION}"}],
        tokenize=False,
        add_generation_prompt=True,
        enable_thinking=thinking,
    )


def boxed_answer(text: str) -> float | None:
    """Number inside the last \\boxed{...}, or None."""
    matches = re.findall(r"\\boxed\{([^{}]*)\}", text)
    if not matches:
        return None
    number = re.sub(r"[^\d.\-]", "", matches[-1])
    try:
        return float(number)
    except ValueError:
        return None


print(repr(prompt("What is 2+3?", thinking=False)[-80:]))
print(boxed_answer("so the total is \\boxed{1,250}."))

In [ ]:
def sync() -> None:
    """Wait for queued device work so timings are real."""
    if DEVICE.type == "cuda":
        torch.cuda.synchronize()
    elif DEVICE.type == "mps":
        torch.mps.synchronize()


@torch.no_grad()
def generate(
    texts: list[str],
    max_new_tokens: int,
    sample: bool = True,
    num_samples: int = 1,
    chunk: int = 32,
    stop_ids: list[int] | None = None,
) -> list[dict]:
    """Batched generation from raw prompt strings.

    Returns one dict per (prompt, sample) with the decoded text and its token count, in
    prompt-major order. Sampling uses Qwen3's recommended thinking-mode settings.
    """
    expanded = [t for t in texts for _ in range(num_samples)]
    out = []
    for start in range(0, len(expanded), chunk):
        batch = tokenizer(
            expanded[start : start + chunk], return_tensors="pt", padding=True
        ).to(DEVICE)
        ids = model.generate(
            **batch,
            max_new_tokens=max_new_tokens,
            do_sample=sample,
            temperature=0.6 if sample else None,
            top_p=0.95 if sample else None,
            top_k=20 if sample else None,
            # extra stop token ids (HF stop_strings re-matches decoded text every step and is slow);
            # without them the model's own end-of-turn tokens apply
            **({"eos_token_id": stop_ids} if stop_ids else {}),
        )
        new = ids[:, batch["input_ids"].size(1) :]
        for row in new:
            row = row[row != tokenizer.pad_token_id]
            out.append(
                {
                    "text": tokenizer.decode(row, skip_special_tokens=False),
                    "tokens": len(row),
                }
            )
    return out


def score(results: list[dict]) -> tuple[float, float]:
    """Accuracy and mean generated tokens of one sample per problem."""
    correct = [boxed_answer(r["text"]) == g for r, g in zip(results, golds)]
    return sum(correct) / len(correct), sum(r["tokens"] for r in results) / len(results)

---
# 3) The math

**Parallel sampling.** Let $p_i$ be the probability that one sample solves problem $i$. With $N$
independent samples:

$$
\text{pass@}N_i = 1 - (1-p_i)^N, \qquad
\text{maj@}N_i = \Pr\big[\text{correct answer is the plurality among } N\big].
$$

pass@$N$ is an **oracle** upper bound: it assumes a perfect verifier. Majority vote needs no
verifier but only helps when the correct answer is the single most likely one
($p_i > \max_{a \neq a^\star} q_i(a)$, where $q_i(a)$ is the probability of wrong answer $a$). Best-of-$N$
with a reward model $r$ picks $\arg\max_j r(x, y_j)$ and sits between the two, depending on how
well $r$ ranks.

**Sequential budget.** For a thinking budget $B$, the forced-stop policy (s1, Muennighoff et al.
2025) generates at most $B$ thinking tokens; if `</think>` has not appeared, it appends an
end-of-thinking cue and `</think>`, then lets the model answer. The minimum-budget policy does
the opposite: when the model emits `</think>` too early, it is replaced by `Wait`, which prompts
the model to re-check. Accuracy as a function of $B$ is the model's **test-time scaling curve**.

**Matched compute.** Comparing strategies fairly means plotting accuracy against total generated
tokens $C$: thinking spends $C \approx B$ in one sample; parallel spends $C \approx N \cdot \bar
L$ with $\bar L$ the mean non-thinking answer length. Snell et al. (2024) showed the best
allocation depends on problem difficulty: easy problems gain from revision, hard ones from search.

---
# 4) From-scratch implementation

## 4.1 Budget forcing

A thinking model left alone decides its own trace length. Budget forcing (s1, Muennighoff et al.
2025) takes that decision away. `think_with_budget` runs two phases explicitly:

```
phase 1:  prompt + <think>  --generate up to B tokens, stop at </think>-->  trace
          if no </think> in trace: trace += END_CUE + "</think>\n\n"    (forced stop)
phase 2:  prompt + trace    --generate up to 192 tokens-->               answer
```

`END_CUE` is the phrase Qwen recommends for this purpose. Phase 2 re-encodes the prompt and trace
instead of reusing the KV cache, which costs one extra prefill and does not change the result.
Thinking mode samples with temperature 0.6, since greedy decoding is known to loop in thinking
mode.

In [ ]:
END_CUE = (
    "\n\nConsidering the limited time by the user, I have to give the solution based on "
    "the thinking directly now.\n"
)


def think_with_budget(questions: list[str], budget: int) -> list[dict]:
    """Thinking capped at `budget` tokens, then a forced answer phase."""
    prompts = [prompt(q, thinking=True) for q in questions]
    traces = [
        r["text"]
        for r in generate(
            prompts, budget, stop_ids=[THINK_END_ID, tokenizer.eos_token_id]
        )
    ]
    closed = [t if THINK_END in t else t + END_CUE + THINK_END + "\n\n" for t in traces]
    closed = [t.split(THINK_END)[0] + THINK_END + "\n\n" for t in closed]
    answers = generate([p + t for p, t in zip(prompts, closed)], 192)
    return [
        {
            "text": t + a["text"],
            "tokens": len(tokenizer(t).input_ids) + a["tokens"],
            "forced": THINK_END not in tr,
        }
        for t, a, tr in zip(closed, answers, traces)
    ]


demo = think_with_budget(questions[:2], budget=64)
print(f"Forced: {[d['forced'] for d in demo]}")
print(demo[0]["text"][-300:])

## 4.2 Thinking on vs off, and the budget curve

Both modes come from the same weights; the switch is the chat template's `enable_thinking` flag.
"Thinking on" here means a 768-token budget, which most traces on this slice fit into and which
keeps the cell to a few minutes; production budgets are 4K–32K tokens.

In [ ]:
THINK_BUDGET = 768
results = {}
start = time.time()
results[False] = generate(
    [prompt(q, thinking=False) for q in questions], max_new_tokens=384
)
acc, tokens = score(results[False])
print(
    f"thinking off: accuracy {acc:.0%}, mean tokens {tokens:6.0f}, {time.time() - start:.0f}s"
)

In [ ]:
start = time.time()
results[True] = think_with_budget(questions, budget=THINK_BUDGET)
acc, tokens = score(results[True])
forced = sum(o["forced"] for o in results[True])
print(
    f"thinking on:  accuracy {acc:.0%}, mean tokens {tokens:6.0f}, {time.time() - start:.0f}s"
)
print(f"Traces force-stopped at {THINK_BUDGET} tokens: {forced}/{NUM_PROBLEMS}")

In [ ]:
budget_curve = {}
for budget in [128, 256, 512]:
    start = time.time()
    outs = think_with_budget(questions, budget)
    acc, tokens = score(outs)
    forced = sum(o["forced"] for o in outs) / len(outs)
    budget_curve[budget] = (acc, tokens)
    print(
        f"budget {budget:>5}: accuracy {acc:.0%}, mean tokens {tokens:6.0f}, "
        f"forced stops {forced:.0%}, {time.time() - start:.0f}s"
    )
budget_curve[THINK_BUDGET] = score(results[True])
print(
    f"budget {THINK_BUDGET:>5}: accuracy {budget_curve[THINK_BUDGET][0]:.0%}, "
    f"mean tokens {budget_curve[THINK_BUDGET][1]:6.0f} (run above)"
)

The minimum-budget direction ("Wait") is s1's other half: when the model emits `</think>` before
a minimum budget, the `</think>` is replaced with `Wait,` and generation continues, which makes
the model re-check its work. s1 showed it raises AIME accuracy for a 32B model. It only matters
with long budgets on hard problems, so it is not run here; it is a three-line change to phase 1
(detect an early `</think>`, strip it, append `Wait,`, generate again).

On this easy slice the budget curve is flat: accuracy stays level from the smallest to the largest
thinking budget, the same as with thinking off. A 0.6B model solves these problems
in its first pass or not at all, so more thinking tokens buy nothing. Test-time scaling curves
rise on problems at the edge of the model's ability (the hard half of GSM8K for this model, AIME
for frontier models).

## 4.3 Parallel sampling and verifiers

$N = 4$ non-thinking samples per problem at temperature 0.6. Four aggregations of the same
samples:

| Aggregator | Uses | Picks |
|---|---|---|
| Majority vote (self-consistency) | Final answers only | Most frequent answer |
| Best-of-N | Reward model score $r(x, y)$ | Highest-scored sample |
| RM-weighted vote | Both | Answer with the largest $\sum_j \exp(r_j)$ |
| pass@N (oracle) | Gold answer | Correct if any sample is correct |

The reward model is `Skywork-Reward-V2-Qwen3-0.6B`, a general-purpose outcome reward model
(sequence classifier on the chat-formatted conversation).

In [ ]:
N = 4
samples = generate(
    [prompt(q, thinking=False) for q in questions], 384, num_samples=N, chunk=48
)
grouped = [samples[i * N : (i + 1) * N] for i in range(NUM_PROBLEMS)]

reward_tokenizer = AutoTokenizer.from_pretrained(REWARD_MODEL_NAME)
reward_model = (
    AutoModelForSequenceClassification.from_pretrained(
        REWARD_MODEL_NAME, dtype=torch.bfloat16
    )
    .to(DEVICE)
    .eval()
)


@torch.no_grad()
def reward(question: str, responses: list[str]) -> list[float]:
    """Scalar reward per response from the outcome reward model."""
    texts = [
        reward_tokenizer.apply_chat_template(
            [
                {"role": "user", "content": question},
                {"role": "assistant", "content": r},
            ],
            tokenize=False,
        )
        for r in responses
    ]
    batch = reward_tokenizer(texts, return_tensors="pt", padding=True).to(DEVICE)
    return reward_model(**batch).logits[:, 0].float().tolist()


rewards = [
    reward(q, [s["text"].split(THINK_END)[-1] for s in g])
    for q, g in zip(questions, grouped)
]
print(f"Rewards for problem 0: {[round(r, 2) for r in rewards[0]]}")
print(
    f"Answers for problem 0: {[boxed_answer(s['text']) for s in grouped[0]]} (gold {golds[0]})"
)

In [ ]:
def aggregate(
    group: list[dict], scores: list[float], k: int
) -> dict[str, float | None]:
    """Answers chosen by each aggregator from the first k samples."""
    answers = [boxed_answer(s["text"]) for s in group[:k]]
    scores = scores[:k]
    valid = [a for a in answers if a is not None]
    weights = Counter()
    for a, s in zip(answers, scores):
        if a is not None:
            weights[a] += math.exp(s)
    return {
        "first sample": answers[0],
        "majority vote": Counter(valid).most_common(1)[0][0] if valid else None,
        "best-of-N (RM)": answers[max(range(k), key=lambda j: scores[j])],
        "RM-weighted vote": weights.most_common(1)[0][0] if weights else None,
    }


parallel_curve = {}
for k in [1, 2, 4]:
    picks = [aggregate(g, r, k) for g, r in zip(grouped, rewards)]
    row = {
        name: sum(p[name] == gold for p, gold in zip(picks, golds)) / NUM_PROBLEMS
        for name in picks[0]
    }
    row["pass@N (oracle)"] = (
        sum(
            any(boxed_answer(s["text"]) == gold for s in g[:k])
            for g, gold in zip(grouped, golds)
        )
        / NUM_PROBLEMS
    )
    tokens = sum(s["tokens"] for g in grouped for s in g[:k]) / NUM_PROBLEMS
    parallel_curve[k] = (row, tokens)
    print(
        f"N={k}: "
        + ", ".join(f"{n} {v:.0%}" for n, v in row.items())
        + f" | {tokens:.0f} tokens/problem"
    )

Parallel sampling is where this model's extra compute pays off. Majority vote over several
samples beats a single sample and approaches the oracle pass@k: whenever the correct answer
appears among the samples it is usually also the most common one. The reward model mostly picks
the same answers as the vote; its advantage shows on harder problems where the correct answer
is a minority.

## 4.4 Sequential revision without feedback

Self-refinement asks the model to review its own non-thinking answer and revise it. With no
external signal (no tests, no verifier), the model must detect its own error. The **flip rates**
show whether revision fixes wrong answers more often than it breaks right ones.

In [ ]:
REVIEW = (
    "Review your solution above carefully. If you find a mistake, correct it. "
    "Put your final answer within \\boxed{}."
)


def revise(questions: list[str], previous: list[dict]) -> list[dict]:
    """One round of self-review as a second user turn."""
    texts = [
        tokenizer.apply_chat_template(
            [
                {"role": "user", "content": f"{q}\n{INSTRUCTION}"},
                {
                    "role": "assistant",
                    "content": p["text"].split(THINK_END)[-1].strip(),
                },
                {"role": "user", "content": REVIEW},
            ],
            tokenize=False,
            add_generation_prompt=True,
            enable_thinking=False,
        )
        for q, p in zip(questions, previous)
    ]
    return generate(texts, 384)


rounds = [[g[0] for g in grouped]]
rounds.append(revise(questions, rounds[0]))
for i in range(1, len(rounds)):
    before = [boxed_answer(r["text"]) == g for r, g in zip(rounds[i - 1], golds)]
    after = [boxed_answer(r["text"]) == g for r, g in zip(rounds[i], golds)]
    fixed = sum(not b and a for b, a in zip(before, after))
    broken = sum(b and not a for b, a in zip(before, after))
    print(
        f"round {i}: accuracy {sum(before) / NUM_PROBLEMS:.0%} -> {sum(after) / NUM_PROBLEMS:.0%}, "
        f"wrong->right {fixed}, right->wrong {broken}"
    )

On a slice this small the effect of a single review round is within noise; on larger studies (Huang et al., 2024) intrinsic self-correction without
feedback is roughly zero-sum on reasoning benchmarks, and helps mainly when the first answer
came out of a careless slip that a second read catches.

## 4.5 Accuracy at matched compute

In [ ]:
plt.figure(figsize=(7, 4))
xs = [budget_curve[b][1] for b in budget_curve]
plt.plot(
    xs,
    [budget_curve[b][0] for b in budget_curve],
    marker="o",
    label="thinking, budget forcing",
)
for name in ["majority vote", "best-of-N (RM)", "pass@N (oracle)"]:
    plt.plot(
        [parallel_curve[k][1] for k in parallel_curve],
        [parallel_curve[k][0][name] for k in parallel_curve],
        marker="s",
        label=f"non-thinking, {name}",
    )
revision_tokens = [
    sum(r["tokens"] for rr in rounds[: i + 1] for r in rr) / NUM_PROBLEMS
    for i in range(len(rounds))
]
revision_acc = [
    sum(boxed_answer(r["text"]) == g for r, g in zip(rr, golds)) / NUM_PROBLEMS
    for rr in rounds
]
plt.plot(revision_tokens, revision_acc, marker="^", label="non-thinking, self-revision")
plt.xscale("log")
plt.xlabel("generated tokens per problem")
plt.ylabel("accuracy")
plt.legend(fontsize=8)
plt.tight_layout()
plt.show()

## 4.6 Overthinking and early exit

A thinking trace often reaches the right answer long before `</think>`; the rest is re-checking.
To find where, the trace is cut at every 192-token boundary and the model is **forced to answer**
from the prefix (append the end cue and `</think>`, then greedy-decode a short answer). The first
cut whose answer equals the final answer marks the point the model had already "decided".

This probe gives an **early-exit rule** that needs no labels: stop thinking as soon as two
consecutive probes return the same answer (answer convergence).

In [ ]:
@torch.no_grad()
def probe_answers(
    question: str, trace: str, every: int = 192
) -> list[tuple[int, float | None]]:
    """Forced answers from trace prefixes at every `every` tokens."""
    ids = tokenizer(trace.split(THINK_END)[0]).input_ids
    cuts = list(range(every, len(ids), every))
    texts = [
        prompt(question, thinking=True)
        + tokenizer.decode(ids[:c])
        + END_CUE
        + THINK_END
        + "\n\n"
        for c in cuts
    ]
    outs = generate(texts, 160, sample=False) if texts else []
    return [(c, boxed_answer(o["text"])) for c, o in zip(cuts, outs)]


def first_stable(probes: list[tuple[int, float | None]]) -> int | None:
    """Token count at which two consecutive probes first agree on a non-empty answer."""
    for (c0, a0), (c1, a1) in zip(probes, probes[1:]):
        if a0 is not None and a0 == a1:
            return c1
    return None


start = time.time()
saved, kept_correct, total_correct = [], 0, 0
for q, r, gold in zip(questions, results[True], golds):
    probes = probe_answers(q, r["text"])
    full_tokens = len(tokenizer(r["text"].split(THINK_END)[0]).input_ids)
    stop = first_stable(probes)
    answer_at_stop = dict(probes).get(stop) if stop else boxed_answer(r["text"])
    saved.append(1 - (stop or full_tokens) / full_tokens)
    total_correct += boxed_answer(r["text"]) == gold
    kept_correct += answer_at_stop == gold
print(
    f"Thinking tokens saved by convergence early exit: {sum(saved) / len(saved):.0%} on average"
)
print(
    f"Correct: full traces {total_correct}, early exit {kept_correct} (of {len(saved)} traces)"
)
print(f"Probe time: {time.time() - start:.0f}s")

In [ ]:
correct_tokens = [
    r["tokens"] for r, g in zip(results[True], golds) if boxed_answer(r["text"]) == g
]
wrong_tokens = [
    r["tokens"] for r, g in zip(results[True], golds) if boxed_answer(r["text"]) != g
]
mean = lambda xs: sum(xs) / len(xs) if xs else float("nan")
print(
    f"Mean thinking-mode tokens: correct answers {mean(correct_tokens):.0f} ({len(correct_tokens)}), "
    f"wrong answers {mean(wrong_tokens):.0f} ({len(wrong_tokens)})"
)

---
# 5) Practical notes

| Control | Where it lives | Notes |
|---|---|---|
| Thinking on/off | `enable_thinking` (Qwen3 template), `/think` `/no_think` soft switches | Same weights; non-thinking for latency-bound or easy traffic |
| Thinking budget | Claude `budget_tokens`, Gemini `thinkingBudget`, s1-style forcing for open models | Budget is a cap; quality degrades sharply below the model's natural length on hard problems |
| Effort level | OpenAI `reasoning_effort` | Trained behaviour, not a hard cap |
| Parallel sampling | `n` in the API, `num_return_sequences` | KV cache of the prompt is shared (prefix caching) so $N$ samples cost far less than $N$ prefills |
| Verifier | Reward model, PRM, unit tests, or a judge model | The aggregator is only as good as the verifier's ranking; reward hacking at large $N$ |

**Failure modes.**

- **Greedy thinking loops.** Use the recommended sampling settings (Qwen3: T=0.6, top-p 0.95,
  top-k 20); repetition in long traces is a common cause of hitting the budget.
- **Forced stops cost accuracy on hard problems.** The answer comes from an unfinished trace.
- **Majority vote saturates** when the model is consistently wrong; only a verifier or more
  sequential compute moves those problems.
- **Self-revision without feedback** is close to zero-sum on its own outputs; it helps when the
  second turn gets new information (test failures, tool output, a verifier's critique).
- **Serving.** Long traces dominate decode time and KV memory; the reasoning trace is parsed out
  by the server (`--reasoning-parser` in vLLM/SGLang) and is usually not returned to the user.

---
# 6) Conclusion

| Built | Takeaway |
|---|---|
| Thinking on/off measurement | One checkpoint, two cost profiles; thinking costs several times the tokens and only pays off on problems near the edge of the model's ability |
| Budget forcing (s1) | A two-phase loop turns any thinking model into one with a hard token budget; accuracy vs budget is the test-time scaling curve |
| Majority vote, best-of-N, weighted vote | For a small model on easy problems, a few votes beat a long trace; the gap to pass@N is the verifier's headroom |
| Self-revision | Without external feedback, gains are small and noisy; feedback (tests, a verifier) is what makes revision reliable |
| Answer-convergence early exit | Many thinking tokens come after the answer is settled; probing finds a label-free stopping point |